# MITRE ATT&CK Gap Analysis Framework

**Purpose:** Automated threat-informed defense gap analysis using MITRE ATT&CK

**Workflow:**
1. Ingest threat actor profiles (MITRE groups, custom layers, parsed threat intel)
2. Weight threats by organizational concern
3. Create composite threat landscape
4. Map defensive coverage (DeTTECT integration)
5. Calculate and visualize gaps
6. Export multi-layered Navigator views and reports

**Output:**
- Multi-tabbed ATT&CK Navigator with threat, coverage, and gap layers
- Prioritized gap analysis report
- Recommendations for improving coverage

---

In [ ]:
"""
Setup and Dependencies
Install required packages and configure environment
"""

# Auto-install dependencies if missing
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    pkg_name = package.replace('-', '_').split('>=')[0].split('==')[0].split('[')[0]
    try:
        __import__(pkg_name)
        return True
    except ImportError:
        print(f"📦 Installing {package}...")
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
            print(f"✅ Installed {package}")
            return True
        except Exception as e:
            print(f"❌ Failed to install {package}: {e}")
            print(f"   Please run: pip install {package}")
            return False

# Required packages (including attack_parser.py dependencies)
required_packages = [
    'pandas>=2.0.0',
    'requests>=2.31.0',
    'pyyaml>=6.0',
    'beautifulsoup4>=4.12.0',
    'lxml',
    'pypdf',  # Required by attack_parser.py for PDF parsing
    'tabulate',  # Required by pandas.to_markdown() for report generation
    'markdown'  # Required for converting markdown reports to HTML
]

print("🔍 Checking dependencies...")
for package in required_packages:
    install_package(package)
print("")

# Standard library imports
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from datetime import datetime
import logging

# Third-party imports
import pandas as pd
import requests
import yaml
from bs4 import BeautifulSoup

# Optional: Jupyter widgets for interactive controls
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, Markdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("⚠️  ipywidgets not available - interactive controls will be limited")

# ============================================================================
# CONFIGURATION: ATT&CK Version
# ============================================================================
# Change this to use a different version of MITRE ATT&CK
# Available versions: 10, 11, 12, 13, 14, 15, 16, 17, 18
# Latest: 18 (as of January 2026)
# ============================================================================
ATTACK_VERSION = "18"

# Directory paths
OUTPUT_DIR = Path("./output")
DATA_DIR = Path("./data")
LAYERS_DIR = Path("./layers")

# Create directories if they don't exist
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
LAYERS_DIR.mkdir(exist_ok=True)

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Environment setup complete")
print(f"📁 Output directory: {OUTPUT_DIR.absolute()}")
print(f"📁 Data directory: {DATA_DIR.absolute()}")
print(f"🎯 ATT&CK Version: {ATTACK_VERSION}")
print(f"   💡 To use a different version, edit ATTACK_VERSION in Cell 1 and restart the kernel")

In [ ]:
"""
Project Configuration
Define scoring scales, color gradients, and project settings
"""

# Scoring Configuration
SCORING_CONFIG = {
    'threat': {
        'min': 1,
        'max': 5,
        'description': 'Threat actor technique usage (1=rare, 5=signature)'
    },
    'coverage': {
        'min': 1,
        'max': 5,
        'description': 'Detection coverage (1=minimal, 5=full automation)'
    },
    'gap': {
        'min': -10,
        'max': 10,
        'description': 'Gap score (negative=over-covered, positive=under-covered)'
    }
}

# Navigator Color Gradients
GRADIENTS = {
    'threat': {
        'colors': ['#ffffff', '#ff6666'],  # White to Red
        'minValue': 1,
        'maxValue': 5
    },
    'coverage': {
        'colors': ['#ffffff', '#6666ff'],  # White to Blue
        'minValue': 1,
        'maxValue': 5
    },
    'gap': {
        'colors': ['#0000ff', '#ffffff', '#ff0000'],  # Blue to White to Red
        'minValue': -10,
        'maxValue': 10
    }
}

# MITRE ATT&CK API Endpoints
ATTACK_API = {
    'stix': f'https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json',
    'groups': f'https://attack.mitre.org/groups/',
}

# Project Metadata
PROJECT_CONFIG = {
    'name': 'Gap Analysis',
    'description': 'Threat-informed defense gap analysis',
    'version': '1.0',
    'created': datetime.now().isoformat()
}

print("✅ Configuration loaded")
print(f"📊 Threat scoring: {SCORING_CONFIG['threat']['min']}-{SCORING_CONFIG['threat']['max']}")
print(f"🛡️  Coverage scoring: {SCORING_CONFIG['coverage']['min']}-{SCORING_CONFIG['coverage']['max']}")
print(f"📈 Gap scoring: {SCORING_CONFIG['gap']['min']}-{SCORING_CONFIG['gap']['max']}")

In [ ]:
"""
Utility Functions
Helper functions used throughout the analysis
"""

def validate_technique_id(tech_id: str) -> bool:
    """Validate ATT&CK technique ID format (T####.### or T####)"""
    import re
    pattern = r'^T\d{4}(\.\d{3})?$'
    return bool(re.match(pattern, tech_id))

def normalize_score(score: float, source_min: float, source_max: float, 
                    target_min: float, target_max: float) -> float:
    """Normalize a score from one range to another"""
    if source_max == source_min:
        return target_min
    normalized = ((score - source_min) / (source_max - source_min)) * (target_max - target_min) + target_min
    return max(target_min, min(target_max, normalized))

def load_json_file(filepath: Path) -> dict:
    """Load JSON file with error handling"""
    try:
        with open(filepath, 'r') as f:
            return json.load(f)
    except Exception as e:
        logger.error(f"Error loading {filepath}: {e}")
        return {}

def save_json_file(data: dict, filepath: Path):
    """Save data as JSON with pretty formatting"""
    try:
        with open(filepath, 'w') as f:
            json.dump(data, f, indent=2)
        logger.info(f"✅ Saved: {filepath}")
    except Exception as e:
        logger.error(f"Error saving {filepath}: {e}")

def create_navigator_layer_template() -> dict:
    """Create an empty ATT&CK Navigator layer structure"""
    return {
        "name": "Layer",
        "versions": {
            "attack": ATTACK_VERSION,
            "navigator": "5.1.0",
            "layer": "4.5"
        },
        "domain": "enterprise-attack",
        "description": "",
        "filters": {
            "platforms": ["Windows", "Linux", "macOS", "Cloud"]
        },
        "sorting": 0,
        "layout": {
            "layout": "side",
            "aggregateFunction": "average",
            "showID": True,
            "showName": True
        },
        "hideDisabled": False,
        "techniques": [],
        "gradient": {
            "colors": ["#ffffff", "#ff6666"],
            "minValue": 0,
            "maxValue": 100
        },
        "legendItems": [],
        "metadata": [],
        "showTacticRowBackground": False,
        "tacticRowBackground": "#dddddd",
        "selectTechniquesAcrossTactics": True,
        "selectSubtechniquesWithParent": False
    }

print("✅ Utility functions loaded")

In [ ]:
"""
MITRE ATT&CK Data Loader
Fetch and cache ATT&CK STIX data for lookups
"""

class MITREAttackLoader:
    def __init__(self, version: str = ATTACK_VERSION, cache_dir: Path = DATA_DIR):
        self.version = version
        self.cache_dir = cache_dir
        self.cache_file = cache_dir / f"attack_v{version}.json"
        self.data = None
        self.techniques_by_id = {}
        self.groups_by_id = {}
        
    def load_attack_data(self, force_refresh: bool = False) -> dict:
        """Load ATT&CK STIX data from cache or API"""
        if not force_refresh and self.cache_file.exists():
            logger.info(f"📦 Loading cached ATT&CK data from {self.cache_file}")
            self.data = load_json_file(self.cache_file)
        else:
            logger.info(f"🌐 Fetching ATT&CK data from MITRE CTI repository...")
            try:
                response = requests.get(ATTACK_API['stix'], timeout=30)
                response.raise_for_status()
                self.data = response.json()
                save_json_file(self.data, self.cache_file)
            except Exception as e:
                logger.error(f"❌ Error fetching ATT&CK data: {e}")
                return {}
        
        # Build lookup dictionaries
        self._build_lookups()
        return self.data
    
    def _build_lookups(self):
        """Build lookup dictionaries for quick access"""
        if not self.data:
            return
        
        for obj in self.data.get('objects', []):
            obj_type = obj.get('type')
            
            # Index techniques
            if obj_type == 'attack-pattern':
                for ext_ref in obj.get('external_references', []):
                    if ext_ref.get('source_name') == 'mitre-attack':
                        tech_id = ext_ref.get('external_id')
                        if tech_id:
                            self.techniques_by_id[tech_id] = obj
            
            # Index groups
            elif obj_type == 'intrusion-set':
                for ext_ref in obj.get('external_references', []):
                    if ext_ref.get('source_name') == 'mitre-attack':
                        group_id = ext_ref.get('external_id')
                        if group_id:
                            self.groups_by_id[group_id] = obj
    
    def get_group_techniques(self, group_id: str) -> Dict[str, List[str]]:
        """Get all techniques used by a group"""
        techniques = {}
        
        if not self.data:
            logger.warning("⚠️  ATT&CK data not loaded")
            return techniques
        
        # Find relationships where this group uses techniques
        for obj in self.data.get('objects', []):
            if obj.get('type') == 'relationship' and obj.get('relationship_type') == 'uses':
                
                # Check if source is our group
                source_ref = obj.get('source_ref', '')
                if source_ref == self.groups_by_id.get(group_id, {}).get('id'):
                    target_ref = obj.get('target_ref', '')
                    
                    # Find the technique this relationship points to
                    for tech_id, tech_obj in self.techniques_by_id.items():
                        if tech_obj.get('id') == target_ref:
                            techniques[tech_id] = {
                                'name': tech_obj.get('name'),
                                'description': obj.get('description', '')
                            }
        
        return techniques
    
    def get_technique_metadata(self, technique_id: str) -> dict:
        """Get metadata for a specific technique"""
        tech_obj = self.techniques_by_id.get(technique_id, {})
        
        # Extract tactics from kill_chain_phases (list of dicts)
        tactics = []
        for phase in tech_obj.get('kill_chain_phases', []):
            if isinstance(phase, dict):
                phase_name = phase.get('phase_name', '')
                if phase_name:
                    tactics.append(phase_name)
        
        return {
            'id': technique_id,
            'name': tech_obj.get('name', 'Unknown'),
            'description': tech_obj.get('description', ''),
            'tactics': tactics,
            'platforms': tech_obj.get('x_mitre_platforms', [])
        }
    
    def list_available_groups(self) -> pd.DataFrame:
        """List all available threat actor groups"""
        groups = []
        for group_id, group_obj in self.groups_by_id.items():
            groups.append({
                'ID': group_id,
                'Name': group_obj.get('name'),
                'Aliases': ', '.join(group_obj.get('aliases', [])),
                'Description': group_obj.get('description', '')[:100] + '...'
            })
        return pd.DataFrame(groups)

# Initialize the loader
attack_loader = MITREAttackLoader()
attack_data = attack_loader.load_attack_data()

print(f"✅ ATT&CK data loaded")
print(f"📊 Techniques: {len(attack_loader.techniques_by_id)}")
print(f"👥 Groups: {len(attack_loader.groups_by_id)}")

In [ ]:
"""
Available MITRE ATT&CK Groups
Browse available threat actor groups for analysis
"""

groups_df = attack_loader.list_available_groups()
print(f"📋 Available Threat Actor Groups: {len(groups_df)}")
print("\nSample groups:")
display(groups_df.head(10))

# Optionally save full list
groups_df.to_csv(OUTPUT_DIR / 'available_groups.csv', index=False)
print(f"\n💾 Full list saved to: {OUTPUT_DIR / 'available_groups.csv'}")

In [ ]:
"""
Navigator Layer Handler
Load, manipulate, and export ATT&CK Navigator layers
"""

class NavigatorLayer:
    def __init__(self, name: str = "Layer", description: str = ""):
        self.layer = create_navigator_layer_template()
        self.layer['name'] = name
        self.layer['description'] = description
        self.techniques = {}  # technique_id: {'score': float, 'comment': str}
    
    def add_technique(self, technique_id: str, score: float, comment: str = "", 
                     enabled: bool = True):
        """Add or update a technique in the layer"""
        if not validate_technique_id(technique_id):
            logger.warning(f"⚠️  Invalid technique ID: {technique_id}")
            return
        
        self.techniques[technique_id] = {
            'score': score,
            'comment': comment,
            'enabled': enabled
        }
    
    def set_gradient(self, gradient_type: str = 'threat'):
        """Set the color gradient for this layer"""
        if gradient_type in GRADIENTS:
            self.layer['gradient'] = GRADIENTS[gradient_type]
        else:
            logger.warning(f"⚠️  Unknown gradient type: {gradient_type}")
    
    def to_json(self) -> dict:
        """Convert to Navigator JSON format with techniques sorted by score (highest first)"""
        # Build techniques array
        techniques_array = []
        for tech_id, tech_data in self.techniques.items():
            tech_entry = {
                'techniqueID': tech_id,
                'score': tech_data['score'],
                'enabled': tech_data.get('enabled', True)
            }
            if tech_data.get('comment'):
                tech_entry['comment'] = tech_data['comment']
            techniques_array.append(tech_entry)
        
        # Sort techniques by score (highest to lowest)
        techniques_array.sort(key=lambda x: x['score'], reverse=True)
        
        self.layer['techniques'] = techniques_array
        return self.layer
    
    @classmethod
    def from_json(cls, data: dict) -> 'NavigatorLayer':
        """Load from Navigator JSON"""
        layer = cls(
            name=data.get('name', 'Layer'),
            description=data.get('description', '')
        )
        layer.layer = data
        
        # Extract techniques
        for tech in data.get('techniques', []):
            tech_id = tech.get('techniqueID')
            if tech_id:
                layer.techniques[tech_id] = {
                    'score': tech.get('score', 0),
                    'comment': tech.get('comment', ''),
                    'enabled': tech.get('enabled', True)
                }
        
        return layer
    
    @classmethod
    def from_file(cls, filepath: Path) -> 'NavigatorLayer':
        """Load from file"""
        data = load_json_file(filepath)
        return cls.from_json(data)
    
    def save(self, filepath: Path):
        """Save layer to file"""
        save_json_file(self.to_json(), filepath)
    
    def get_techniques_dataframe(self) -> pd.DataFrame:
        """Get techniques as a pandas DataFrame"""
        rows = []
        for tech_id, tech_data in self.techniques.items():
            metadata = attack_loader.get_technique_metadata(tech_id)
            rows.append({
                'Technique ID': tech_id,
                'Name': metadata['name'],
                'Score': tech_data['score'],
                'Tactics': ', '.join(metadata['tactics']),
                'Comment': tech_data.get('comment', '')
            })
        return pd.DataFrame(rows)

print("✅ NavigatorLayer class loaded")

In [ ]:
"""
Attack Parser Integration
Wrapper for attack_parser.py tool (optional)
"""

class AttackParserIntegration:
    def __init__(self, parser_script_path: str = "./attack_parser.py"):
        self.parser_path = Path(parser_script_path)
        self.is_available = self.parser_path.exists()
        
        if not self.is_available:
            logger.warning(f"⚠️  Attack parser not found at {self.parser_path}")
            logger.warning("   This is optional - only needed if using 'parsed' threat sources")
            logger.warning("   Download from: https://github.com/mjmcphee/attack_parser")
            logger.warning("   Place attack_parser.py in the project root directory")
    
    def parse_threat_report(self, source: str, source_type: str = 'url',
                           output_name: str = None) -> NavigatorLayer:
        """
        Parse a threat report using attack_parser
        
        Args:
            source: URL, file path, or text content
            source_type: 'url', 'file', or 'text'
            output_name: Optional custom name for the output layer
        
        Returns:
            NavigatorLayer object or None if parser unavailable
        """
        import subprocess
        
        if not self.is_available:
            raise FileNotFoundError(
                f"attack_parser.py not found at {self.parser_path}\n"
                f"Download from: https://github.com/mjmcphee/attack_parser\n"
                f"Place it in: {self.parser_path.parent.absolute()}"
            )
        
        # Generate output filename
        if output_name:
            output_file = LAYERS_DIR / f"{output_name}.json"
        else:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_file = LAYERS_DIR / f"parsed_layer_{timestamp}.json"
        
        # Build command
        cmd = [
            'python', str(self.parser_path),
            f'--{source_type}', source,
            '--output', str(output_file),
            '--score', '5'  # Default to max score for parsed intel
        ]
        
        logger.info(f"🔍 Parsing threat report: {source}")
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
            
            if result.returncode == 0:
                logger.info(f"✅ Successfully parsed threat report")
                return NavigatorLayer.from_file(output_file)
            else:
                logger.error(f"❌ Parser error: {result.stderr}")
                return None
                
        except subprocess.TimeoutExpired:
            logger.error("❌ Parser timed out")
            return None
        except Exception as e:
            logger.error(f"❌ Error running parser: {e}")
            return None
    
    def batch_parse(self, sources: List[Tuple[str, str]]) -> List[NavigatorLayer]:
        """
        Parse multiple sources
        
        Args:
            sources: List of (source, source_type) tuples
        
        Returns:
            List of NavigatorLayer objects
        """
        layers = []
        for i, (source, source_type) in enumerate(sources):
            layer = self.parse_threat_report(
                source, 
                source_type, 
                output_name=f"parsed_{i+1}"
            )
            if layer:
                layers.append(layer)
        return layers

# Initialize parser integration
parser = AttackParserIntegration()

if parser.is_available:
    print("✅ Attack parser integration ready")
    print(f"   📄 Parser location: {parser.parser_path.absolute()}")
else:
    print("⚠️  Attack parser NOT available")
    print("   ℹ️  Only needed if using 'parsed' threat intelligence sources")
    print("   📥 Download from: https://github.com/mjmcphee/attack_parser")
    print(f"   📂 Place file in: {parser.parser_path.parent.absolute()}")

# ============================================================================
# AUTO-DOWNLOAD ATTACK_PARSER.PY
# ============================================================================

"""
Auto-Install Attack Parser (Optional)
Downloads attack_parser.py if not present - only needed for 'parsed' threat sources
"""

ATTACK_PARSER_URL = "https://raw.githubusercontent.com/mjmcphee/attack_parser/main/attack_parser.py"
ATTACK_PARSER_PATH = Path("./attack_parser.py")

def download_attack_parser(force: bool = False) -> bool:
    """
    Download attack_parser.py from GitHub if not present
    
    Args:
        force: Force re-download even if file exists
    
    Returns:
        True if successful, False otherwise
    """
    if ATTACK_PARSER_PATH.exists() and not force:
        print(f"✅ attack_parser.py already exists")
        return True
    
    print(f"📥 Downloading attack_parser.py from GitHub...")
    try:
        response = requests.get(ATTACK_PARSER_URL, timeout=30)
        response.raise_for_status()
        
        with open(ATTACK_PARSER_PATH, 'w') as f:
            f.write(response.text)
        
        print(f"✅ Successfully downloaded attack_parser.py")
        print(f"   Location: {ATTACK_PARSER_PATH.absolute()}")
        return True
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Failed to download attack_parser.py: {e}")
        print(f"   You can manually download from: {ATTACK_PARSER_URL}")
        return False
    except Exception as e:
        print(f"❌ Error saving file: {e}")
        return False

# Auto-download if needed
# Set AUTO_DOWNLOAD = False if you don't want automatic downloads
AUTO_DOWNLOAD = True

if AUTO_DOWNLOAD:
    download_attack_parser()
else:
    if ATTACK_PARSER_PATH.exists():
        print(f"✅ attack_parser.py found at {ATTACK_PARSER_PATH.absolute()}")
    else:
        print(f"ℹ️  attack_parser.py not found (auto-download disabled)")
        print(f"   Run: download_attack_parser() to download manually")
        print(f"   Or download from: {ATTACK_PARSER_URL}")

print("\n💡 Note: attack_parser.py is only needed if using 'parsed' threat sources")
# Reinitialize parser to detect the newly downloaded file
parser = AttackParserIntegration()
if parser.is_available:
    print(f"\n✅ Parser reinitialized and ready")
    print(f"   📄 Parser location: {parser.parser_path.absolute()}")


## 🎯 Configure Threat Actors

Define which threat actors your organization is concerned about and assign weights (0.0 to 1.0).

**Options:**
1. **MITRE Groups:** Use standard ATT&CK group profiles (e.g., APT29, FIN7)
2. **Custom Layers:** Load existing Navigator layers from `./layers/` directory
3. **Parsed Intel:** Use attack_parser to extract techniques from threat reports

**Using Parsed Sources:**

To use `'source': 'parsed'` for blog posts or threat reports:
1. Download `attack_parser.py` from https://github.com/mjmcphee/attack_parser
2. Place it in this project directory (same folder as this notebook)
3. Configure the URL in the threat actor config below

If you don't have attack_parser.py, comment out any parsed sources or the notebook will fail at the next step.

In [ ]:
"""
Threat Actor Configuration
Define threat actors and their organizational weights
"""

# Threat actor configuration
# Format: {
#   'name': str,
#   'source': 'mitre' | 'file' | 'parsed',
#   'identifier': group_id | filepath | url,
#   'weight': float (0.0 to 1.0)
# }

THREAT_ACTORS = [
    {
        'name': 'APT29 (Cozy Bear)',
        'source': 'mitre',
        'identifier': 'G0016',  # MITRE group ID
        'weight': 0.9,
        'description': 'Russian state-sponsored threat actor, LOLBINs focus - FSB'
    },
    {
        'name': 'APT28 (Fancy Bear)',
        'source': 'mitre',
        'identifier': 'G0007',
        'weight': 0.7,
        'description': 'Russian state-sponsored threat actor - GRU'
    },
    {
        'name': 'Operation Ghost',
        'source': 'mitre',
        'identifier': 'C0023',
        'weight': 0.5,
        'description': 'APT29 campaign against foreign agencies in EU/US'
    },
    {
        'name': 'Turla',
        'source': 'mitre',
        'identifier': 'G0010',
        'weight': 0.5,
        'description': 'Russian Cyber Espionage, malware focus - FSB'
    },
    {
        'name': 'APT35 (Magic Hound/Charming Kitten)',
        'source': 'mitre',
        'identifier': 'G0059',
        'weight': 0.3,
        'description': 'Iranian state-sponsored threat actor targeting EU - GRU'
    },
    # Example: Custom layer file
    {
        'name': 'Insider Threats - Aggregate',
        'source': 'file',
        'identifier': './layers/insider_threat_knowledge_basev2.json',
        'weight': 1.0,
        'description': 'Insider Threats per CTID and recent events'
    },
    
    # Example: Parse recent threat intel
    {
        'name': 'APT29 Diplomatic Phishing',
        'source': 'parsed',
        'identifier': 'https://cloud.google.com/blog/topics/threat-intelligence/apt29-evolving-diplomatic-phishing',
        'weight': 0.6,
        'description': 'APT29 diplomatic phishing campaign'
    },
    {
        'name': 'Turla (Polish NGO specifics)',
        'source': 'parsed',
        'identifier': 'https://www.darkreading.com/cyberattacks-data-breaches/russian-apt-turla-novel-backdoor-malware-polish-ngos',
        'weight': 0.8,
        'description': 'APT29 diplomatic phishing campaign'
    }
]


print(f"✅ Configured {len(THREAT_ACTORS)} threat actors:")
for actor in THREAT_ACTORS:
    print(f"   • {actor['name']} (weight: {actor['weight']})")

In [ ]:
"""
Load Threat Actor Layers
Load techniques for each configured threat actor
"""

class ThreatActor:
    def __init__(self, name: str, weight: float, layer: NavigatorLayer, 
                 description: str = ""):
        self.name = name
        self.weight = weight
        self.layer = layer
        self.description = description
        self.techniques = layer.techniques  # Dict[tech_id: score]
    
    def get_summary(self) -> dict:
        """Get summary statistics"""
        return {
            'name': self.name,
            'weight': self.weight,
            'technique_count': len(self.techniques),
            'avg_score': sum(t['score'] for t in self.techniques.values()) / len(self.techniques) if self.techniques else 0
        }

def load_threat_actor(config: dict) -> ThreatActor:
    """Load a threat actor based on configuration"""
    name = config['name']
    source = config['source']
    identifier = config['identifier']
    weight = config['weight']
    description = config.get('description', '')
    
    logger.info(f"📥 Loading: {name}")
    
    if source == 'mitre':
        # Load MITRE group
        group_id = identifier
        techniques = attack_loader.get_group_techniques(group_id)
        
        layer = NavigatorLayer(
            name=name,
            description=f"MITRE ATT&CK Group {group_id}"
        )
        
        # Add techniques with default score of 3 (moderate)
        # You could enhance this by analyzing frequency in group descriptions
        for tech_id in techniques.keys():
            layer.add_technique(tech_id, score=3, comment=f"Used by {name}")
        
    elif source == 'file':
        # Load from file
        filepath = Path(identifier)
        if not filepath.exists():
            raise FileNotFoundError(f"Layer file not found: {filepath}")
        layer = NavigatorLayer.from_file(filepath)
        layer.layer['name'] = name
        
    elif source == 'parsed':
        # Check if parser is available first
        if not parser.is_available:
            raise FileNotFoundError(
                f"Cannot load '{name}': attack_parser.py is required for 'parsed' sources.\n"
                f"Download from: https://github.com/mjmcphee/attack_parser\n"
                f"Place it in: {parser.parser_path.parent.absolute()}/attack_parser.py\n"
                f"OR comment out this threat actor in Cell 8 and re-run."
            )
        
        # Parse using attack_parser
        url = identifier
        layer = parser.parse_threat_report(url, 'url', output_name=name.replace(' ', '_'))
        if not layer:
            raise ValueError(f"Failed to parse threat report: {url}")
        layer.layer['name'] = name
    
    else:
        raise ValueError(f"Unknown source type: {source}")
    
    logger.info(f"   ✅ Loaded {len(layer.techniques)} techniques")
    
    return ThreatActor(name, weight, layer, description)

# Load all configured threat actors
threat_actors = []
failed_actors = []

for config in THREAT_ACTORS:
    try:
        actor = load_threat_actor(config)
        threat_actors.append(actor)
    except FileNotFoundError as e:
        # Special handling for missing parser
        logger.error(f"❌ Cannot load {config['name']}: {e}")
        failed_actors.append(config['name'])
    except Exception as e:
        logger.error(f"❌ Failed to load {config['name']}: {e}")
        failed_actors.append(config['name'])

print(f"\n✅ Successfully loaded {len(threat_actors)}/{len(THREAT_ACTORS)} threat actors")

if failed_actors:
    print(f"\n⚠️  Failed to load {len(failed_actors)} threat actor(s):")
    for name in failed_actors:
        print(f"   • {name}")
    print("\n💡 Tip: Comment out failed actors in Cell 8 if you want to proceed without them")

# Check if we have at least one actor
if not threat_actors:
    raise ValueError(
        "No threat actors were successfully loaded!\n"
        "Please check your configuration in Cell 8 and ensure:\n"
        "  - MITRE group IDs are valid\n"
        "  - Custom layer files exist\n"
        "  - attack_parser.py is available for parsed sources"
    )

# Display summary
summary_data = [actor.get_summary() for actor in threat_actors]
summary_df = pd.DataFrame(summary_data)
display(summary_df)

In [ ]:
"""
Composite Threat Matrix
Combine all threat actors into weighted composite
"""

class ThreatCompositor:
    def __init__(self, actors: List[ThreatActor]):
        self.actors = actors
        self.composite_scores = {}
    
    def calculate_composite(self) -> Dict[str, float]:
        """
        Calculate weighted composite threat scores
        
        For each technique:
        - Find all actors that use it
        - Weight each actor's score by their org weight
        - Average the weighted scores
        """
        # Collect all unique techniques
        all_techniques = set()
        for actor in self.actors:
            all_techniques.update(actor.techniques.keys())
        
        logger.info(f"🔢 Calculating composite for {len(all_techniques)} unique techniques")
        
        # Calculate weighted score for each technique
        for tech_id in all_techniques:
            weighted_scores = []
            
            for actor in self.actors:
                if tech_id in actor.techniques:
                    tech_score = actor.techniques[tech_id]['score']
                    weighted_score = tech_score * actor.weight
                    weighted_scores.append((weighted_score, actor.weight, actor.name))
            
            if weighted_scores:
                # Weighted average: sum(score * weight) / sum(weights)
                total_weighted = sum(ws[0] for ws in weighted_scores)
                total_weight = sum(ws[1] for ws in weighted_scores)
                
                avg_score = total_weighted / total_weight if total_weight > 0 else 0
                
                # Normalize to 1-5 scale
                normalized_score = max(1, min(5, avg_score))
                
                self.composite_scores[tech_id] = {
                    'score': normalized_score,
                    'actors': [ws[2] for ws in weighted_scores],
                    'raw_scores': [ws[0] for ws in weighted_scores]
                }
        
        return self.composite_scores
    
    def generate_layer(self) -> NavigatorLayer:
        """Generate Navigator layer for composite threat"""
        layer = NavigatorLayer(
            name="Composite Threat Landscape",
            description=f"Weighted aggregate of {len(self.actors)} threat actors"
        )
        layer.set_gradient('threat')
        
        for tech_id, data in self.composite_scores.items():
            actors_list = ', '.join(data['actors'])
            layer.add_technique(
                tech_id,
                score=data['score'],
                comment=f"Used by: {actors_list}"
            )
        
        return layer
    
    def get_summary_dataframe(self) -> pd.DataFrame:
        """Get composite scores as DataFrame"""
        rows = []
        for tech_id, data in self.composite_scores.items():
            metadata = attack_loader.get_technique_metadata(tech_id)
            rows.append({
                'Technique ID': tech_id,
                'Name': metadata['name'],
                'Composite Score': round(data['score'], 2),
                'Actor Count': len(data['actors']),
                'Actors': ', '.join(data['actors']),
                'Tactics': ', '.join(metadata['tactics'])
            })
        
        df = pd.DataFrame(rows)
        return df.sort_values('Composite Score', ascending=False)

# Calculate composite threat
compositor = ThreatCompositor(threat_actors)
composite_scores = compositor.calculate_composite()
composite_layer = compositor.generate_layer()

print(f"✅ Composite threat calculated")
print(f"📊 Total unique techniques: {len(composite_scores)}")

# Display top threats
composite_df = compositor.get_summary_dataframe()
print("\n🔥 Top 10 Threat Techniques:")
display(composite_df.head(10))

# Save composite layer
composite_layer.save(OUTPUT_DIR / 'composite_threat.json')
print(f"\n💾 Composite threat layer saved")

In [ ]:
"""
DeTTECT Coverage Loader
Load and convert DeTTECT YAML to coverage scores
"""

class DeTTECTLoader:
    def __init__(self, yaml_path: Path):
        self.yaml_path = yaml_path
        self.coverage_data = {}
    
    def load_yaml(self) -> dict:
        """Load DeTTECT YAML file"""
        if not self.yaml_path.exists():
            raise FileNotFoundError(f"DeTTECT file not found: {self.yaml_path}")
        
        with open(self.yaml_path, 'r') as f:
            data = yaml.safe_load(f)
        
        return data
    
    def extract_coverage_scores(self) -> Dict[str, int]:
        """
        Extract coverage scores from DeTTECT data
        
        DeTTECT uses 0-5 scale for both visibility and detection
        Real DeTTECT format uses score_logbook with dated entries
        We'll combine them: coverage_score = max(visibility, detection)
        """
        data = self.load_yaml()
        
        # DeTTECT structure: techniques list with visibility/detection sub-lists
        for technique in data.get('techniques', []):
            tech_id = technique.get('technique_id')
            if not tech_id:
                continue
            
            # Get visibility scores from score_logbook
            visibility_scores = []
            for vis in technique.get('visibility', []):
                # score_logbook contains dated entries
                score_logbook = vis.get('score_logbook', [])
                if score_logbook:
                    # Get the most recent score (last entry in list)
                    latest_entry = score_logbook[-1]
                    score = latest_entry.get('score', 0)
                    # Ignore -1 scores (not applicable)
                    if isinstance(score, int) and score >= 0:
                        visibility_scores.append(score)
                
                # Fallback: check for simple 'score' field (old format)
                if 'score' in vis and not score_logbook:
                    score = vis.get('score', 0)
                    if isinstance(score, int) and score >= 0:
                        visibility_scores.append(score)
            
            # Get detection scores from score_logbook
            detection_scores = []
            for det in technique.get('detection', []):
                # score_logbook contains dated entries
                score_logbook = det.get('score_logbook', [])
                if score_logbook:
                    # Get the most recent score (last entry in list)
                    latest_entry = score_logbook[-1]
                    score = latest_entry.get('score', 0)
                    # Ignore -1 scores (not applicable)
                    if isinstance(score, int) and score >= 0:
                        detection_scores.append(score)
                
                # Fallback: check for simple 'score' field (old format)
                if 'score' in det and not score_logbook:
                    score = det.get('score', 0)
                    if isinstance(score, int) and score >= 0:
                        detection_scores.append(score)
            
            # Combine: take max of visibility and detection across all platforms
            max_visibility = max(visibility_scores) if visibility_scores else 0
            max_detection = max(detection_scores) if detection_scores else 0
            
            coverage_score = max(max_visibility, max_detection)
            
            # Normalize to 1-5 (DeTTECT uses 0-5, we use 1-5)
            if coverage_score > 0:
                self.coverage_data[tech_id] = max(1, min(5, coverage_score))
        
        return self.coverage_data
    
    def generate_layer(self) -> NavigatorLayer:
        """Generate Navigator layer for coverage"""
        layer = NavigatorLayer(
            name="Detection Coverage",
            description="Current detection and visibility coverage"
        )
        layer.set_gradient('coverage')
        
        for tech_id, score in self.coverage_data.items():
            layer.add_technique(
                tech_id,
                score=score,
                comment=f"Coverage score: {score}/5"
            )
        
        return layer

# ============================================================================
# INTERACTIVE FILE PICKER
# ============================================================================
print("📂 DeTTECT YAML File Selection")
print("=" * 60)

# Find all YAML files in data directory and subdirectories
yaml_files = []
if DATA_DIR.exists():
    for ext in ['*.yaml', '*.yml']:
        yaml_files.extend(DATA_DIR.glob(ext))
        yaml_files.extend(DATA_DIR.glob(f'**/{ext}'))  # Include subdirectories

# Remove duplicates and sort
yaml_files = sorted(set(yaml_files))

if yaml_files and WIDGETS_AVAILABLE:
    print(f"Found {len(yaml_files)} YAML file(s) in {DATA_DIR}:")
    print()
    
    # Create dropdown with file options
    file_options = [('(Create sample file)', None)] + [(f.name, f) for f in yaml_files]
    
    file_selector = widgets.Dropdown(
        options=file_options,
        description='Select file:',
        disabled=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    
    # Display the selector
    display(file_selector)
    
    # Wait a moment for user interaction
    import time
    time.sleep(0.1)
    
    # Get selected file
    if file_selector.value is not None:
        DETTECT_YAML_PATH = file_selector.value
        print(f"\n✅ Selected: {DETTECT_YAML_PATH.name}")
    else:
        print(f"\n💡 No file selected, will create sample file")
        DETTECT_YAML_PATH = DATA_DIR / 'dettect_coverage.yaml'

elif yaml_files:
    # Widgets not available, list files and use first one or default
    print(f"Found {len(yaml_files)} YAML file(s):")
    for i, f in enumerate(yaml_files, 1):
        print(f"   {i}. {f.name}")
    
    # Auto-select first YAML file found
    DETTECT_YAML_PATH = yaml_files[0]
    print(f"\n✅ Auto-selected: {DETTECT_YAML_PATH.name}")
    print(f"💡 To use a different file, edit DETTECT_YAML_PATH in this cell")

else:
    # No YAML files found
    print(f"No YAML files found in {DATA_DIR}")
    print(f"💡 Place your DeTTECT YAML file in the data directory")
    DETTECT_YAML_PATH = DATA_DIR / 'dettect_coverage.yaml'

print()
print(f"📍 Using: {DETTECT_YAML_PATH}")
print(f"   Absolute path: {DETTECT_YAML_PATH.absolute()}")
print()

# Create sample file if needed
if not DETTECT_YAML_PATH.exists():
    print(f"⚠️  File doesn't exist. Creating sample file...")
    
    sample_dettect = {
        'version': 1.2,
        'file_type': 'technique-administration',
        'name': 'Sample Coverage',
        'domain': 'enterprise-attack',
        'platform': ['Windows'],
        'techniques': [
            {
                'technique_id': 'T1566.001',
                'technique_name': 'Phishing: Spearphishing Attachment',
                'visibility': [
                    {
                        'applicable_to': ['Windows'],
                        'score_logbook': [
                            {'date': '2025-01-01', 'score': 3, 'comment': 'Email gateway logs'}
                        ]
                    }
                ],
                'detection': [
                    {
                        'applicable_to': ['Windows'],
                        'score_logbook': [
                            {'date': '2025-01-01', 'score': 2, 'comment': 'SIEM alerts'}
                        ]
                    }
                ]
            },
            {
                'technique_id': 'T1059.001',
                'technique_name': 'PowerShell',
                'visibility': [
                    {
                        'applicable_to': ['Windows'],
                        'score_logbook': [
                            {'date': '2025-01-01', 'score': 4, 'comment': 'PowerShell logging'}
                        ]
                    }
                ],
                'detection': [
                    {
                        'applicable_to': ['Windows'],
                        'score_logbook': [
                            {'date': '2025-01-01', 'score': 3, 'comment': 'EDR detection'}
                        ]
                    }
                ]
            }
        ]
    }
    
    DETTECT_YAML_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(DETTECT_YAML_PATH, 'w') as f:
        yaml.dump(sample_dettect, f)
    
    print(f"   ✅ Created: {DETTECT_YAML_PATH}")
    print(f"   📝 Replace this with your actual DeTTECT export")

print()

# Load coverage data
try:
    dettect_loader = DeTTECTLoader(DETTECT_YAML_PATH)
    coverage_scores = dettect_loader.extract_coverage_scores()
    coverage_layer = dettect_loader.generate_layer()
    
    print(f"✅ Coverage data loaded successfully")
    print(f"🛡️  Techniques with coverage: {len(coverage_scores)}")
    
    # Show some sample scores for verification
    if coverage_scores:
        print(f"\n📊 Sample coverage scores:")
        for tech_id, score in list(coverage_scores.items())[:5]:
            metadata = attack_loader.get_technique_metadata(tech_id)
            print(f"   {tech_id}: {metadata['name'][:40]:40} → Score: {score}/5")
        if len(coverage_scores) > 5:
            print(f"   ... and {len(coverage_scores) - 5} more")
    
    # Save coverage layer
    coverage_layer.save(OUTPUT_DIR / 'coverage.json')
    
except Exception as e:
    logger.error(f"❌ Error loading DeTTECT data: {e}")
    print(f"⚠️  Could not load DeTTECT file. Please check:")
    print(f"   • File is valid YAML format")
    print(f"   • File contains 'techniques' list")
    print(f"   • Techniques have visibility/detection with score_logbook")
    import traceback
    print(f"\nError details:")
    traceback.print_exc()
    coverage_scores = {}
    coverage_layer = None

In [ ]:
"""
Gap Analysis
Calculate gaps between threat landscape and coverage
"""

class GapAnalyzer:
    def __init__(self, threat_scores: Dict[str, float], coverage_scores: Dict[str, int]):
        self.threat_scores = threat_scores
        self.coverage_scores = coverage_scores
        self.gap_scores = {}
    
    def calculate_gaps(self) -> Dict[str, float]:
        """
        Calculate gap scores
        
        Gap = Threat Score - Coverage Score
        Range: -10 to +10
        - Positive (red): Under-covered, needs improvement
        - Negative (blue): Over-covered relative to threat
        - Zero (white): Balanced coverage
        """
        # Get all techniques from both sources
        all_techniques = set(self.threat_scores.keys()) | set(self.coverage_scores.keys())
        
        for tech_id in all_techniques:
            threat_score = self.threat_scores.get(tech_id, 0)
            coverage_score = self.coverage_scores.get(tech_id, 0)
            
            # Calculate raw gap
            raw_gap = threat_score - coverage_score
            
            # Normalize to -10 to +10 scale
            # Threat is 1-5, Coverage is 1-5, so raw gap is -4 to +4
            # Scale to -10 to +10
            normalized_gap = (raw_gap / 4) * 10
            
            # Categorize
            if normalized_gap >= 5:
                category = 'critical'
            elif normalized_gap >= 2.5:
                category = 'high'
            elif normalized_gap >= 0:
                category = 'medium'
            elif normalized_gap >= -2.5:
                category = 'low'
            else:
                category = 'over-covered'
            
            self.gap_scores[tech_id] = {
                'gap_score': normalized_gap,
                'threat_score': threat_score,
                'coverage_score': coverage_score,
                'category': category
            }
        
        return self.gap_scores
    
    def generate_layer(self) -> NavigatorLayer:
        """Generate Navigator layer for gaps"""
        layer = NavigatorLayer(
            name="Gap Analysis",
            description="Coverage gaps (red = needs improvement, blue = well-covered)"
        )
        layer.set_gradient('gap')
        
        for tech_id, data in self.gap_scores.items():
            comment = f"Threat: {data['threat_score']:.1f}, Coverage: {data['coverage_score']}, Gap: {data['gap_score']:.1f}"
            layer.add_technique(
                tech_id,
                score=data['gap_score'],
                comment=comment
            )
        
        return layer
    
    def get_prioritized_gaps(self, top_n: int = 20) -> pd.DataFrame:
        """Get top priority gaps as DataFrame"""
        rows = []
        for tech_id, data in self.gap_scores.items():
            # Only include positive gaps (under-covered)
            if data['gap_score'] <= 0:
                continue
            
            metadata = attack_loader.get_technique_metadata(tech_id)
            rows.append({
                'Priority': int(data['gap_score'] * 10),  # 0-100 scale
                'Technique ID': tech_id,
                'Name': metadata['name'],
                'Threat Score': round(data['threat_score'], 1),
                'Coverage Score': data['coverage_score'],
                'Gap Score': round(data['gap_score'], 1),
                'Category': data['category'].upper(),
                'Tactics': ', '.join(metadata['tactics'][:2])  # Limit display
            })
        
        df = pd.DataFrame(rows)
        if not df.empty:
            df = df.sort_values('Gap Score', ascending=False).head(top_n)
        return df
    
    def get_statistics(self) -> dict:
        """Get summary statistics"""
        if not self.gap_scores:
            return {}
        
        gap_values = [data['gap_score'] for data in self.gap_scores.values()]
        positive_gaps = [g for g in gap_values if g > 0]
        
        categories = {}
        for data in self.gap_scores.values():
            cat = data['category']
            categories[cat] = categories.get(cat, 0) + 1
        
        return {
            'total_techniques': len(self.gap_scores),
            'avg_gap': sum(gap_values) / len(gap_values),
            'max_gap': max(gap_values),
            'min_gap': min(gap_values),
            'under_covered_count': len(positive_gaps),
            'under_covered_pct': (len(positive_gaps) / len(gap_values)) * 100,
            'categories': categories
        }

# Calculate gaps
gap_analyzer = GapAnalyzer(
    threat_scores={tech_id: data['score'] for tech_id, data in composite_scores.items()},
    coverage_scores=coverage_scores
)

gap_scores = gap_analyzer.calculate_gaps()
gap_layer = gap_analyzer.generate_layer()

print(f"✅ Gap analysis complete")

# Statistics
stats = gap_analyzer.get_statistics()
print(f"\n📊 Gap Analysis Statistics:")
print(f"   Total techniques analyzed: {stats.get('total_techniques', 0)}")
print(f"   Average gap score: {stats.get('avg_gap', 0):.2f}")
print(f"   Under-covered techniques: {stats.get('under_covered_count', 0)} ({stats.get('under_covered_pct', 0):.1f}%)")

print(f"\n📈 Gap Categories:")
for category, count in stats.get('categories', {}).items():
    print(f"   {category}: {count}")

# Display prioritized gaps
print(f"\n🎯 Top Priority Gaps:")
priority_gaps = gap_analyzer.get_prioritized_gaps(top_n=15)
display(priority_gaps)

# Save gap layer
gap_layer.save(OUTPUT_DIR / 'gap_analysis.json')
print(f"\n💾 Gap analysis layer saved")

# ============================================================================
# GAP ANALYSIS
# ============================================================================

"""
Multi-Tab Navigator Export
Generate HTML with link to open all layers in Navigator
"""

import base64

def create_multi_tab_navigator(layers: Dict[str, Path], output_file: Path):
    """
    Create HTML page with button to open all layers in Navigator as tabs
    
    Args:
        layers: Dict of {tab_name: layer_filepath}
        output_file: Output HTML file path
    """
    
    html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>ATT&CK Gap Analysis - Layer Viewer</title>
    <style>
        body {{ 
            font-family: Arial, sans-serif; 
            margin: 0; 
            padding: 40px;
            background: #f5f5f5;
            max-width: 1200px;
            margin: 0 auto;
        }}
        .header {{ 
            background: white;
            padding: 30px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            margin-bottom: 30px;
        }}
        .header h1 {{ 
            margin: 0 0 10px 0; 
            color: #333; 
        }}
        .header p {{ 
            color: #666; 
            margin: 5px 0; 
        }}
        .main-action {{
            background: white;
            padding: 30px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            margin-bottom: 30px;
            text-align: center;
        }}
        .main-action h2 {{
            margin: 0 0 15px 0;
            color: #333;
        }}
        .main-action p {{
            color: #666;
            margin: 0 0 20px 0;
        }}
        .btn-large {{
            display: inline-block;
            padding: 16px 32px;
            background: #007bff;
            color: white;
            text-decoration: none;
            border-radius: 8px;
            font-weight: 600;
            font-size: 18px;
            transition: background 0.2s, transform 0.2s;
        }}
        .btn-large:hover {{
            background: #0056b3;
            transform: translateY(-2px);
        }}
        .layer-list {{
            background: white;
            padding: 30px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            margin-bottom: 30px;
        }}
        .layer-list h3 {{
            margin: 0 0 20px 0;
            color: #333;
        }}
        .layer-item {{
            padding: 15px;
            border-left: 4px solid #007bff;
            background: #f8f9fa;
            margin-bottom: 10px;
            border-radius: 4px;
        }}
        .layer-item h4 {{
            margin: 0 0 5px 0;
            color: #333;
            font-size: 16px;
        }}
        .layer-item p {{
            margin: 0;
            color: #666;
            font-size: 14px;
        }}
        .layer-item .stats {{
            margin-top: 8px;
            font-size: 13px;
            color: #888;
        }}
        .instructions {{
            background: #d1ecf1;
            border: 1px solid #bee5eb;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 30px;
        }}
        .instructions h3 {{
            margin: 0 0 10px 0;
            color: #0c5460;
        }}
        .instructions ul {{
            margin: 10px 0;
            padding-left: 20px;
            color: #0c5460;
        }}
        .instructions li {{
            margin: 5px 0;
        }}
        .download-section {{
            background: white;
            padding: 25px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            margin-bottom: 30px;
        }}
        .download-section h3 {{
            margin: 0 0 15px 0;
            color: #333;
        }}
        .btn {{
            display: inline-block;
            padding: 10px 20px;
            background: #6c757d;
            color: white;
            text-decoration: none;
            border-radius: 6px;
            font-weight: 500;
            transition: background 0.2s;
            margin-right: 10px;
            margin-bottom: 10px;
            font-size: 14px;
        }}
        .btn:hover {{
            background: #545b62;
        }}
        .footer {{
            text-align: center;
            color: #888;
            font-size: 14px;
            margin-top: 40px;
        }}
    </style>
</head>
<body>
    <div class="header">
        <h1>🎯 ATT&CK Gap Analysis</h1>
        <p>Multi-layer threat landscape and coverage analysis</p>
        <p><strong>Generated:</strong> {timestamp}</p>
    </div>
    
    <div class="main-action">
        <h2>🚀 Open All Layers in Navigator</h2>
        <p>Click below to open all {layer_count} layers as tabs in a single ATT&CK Navigator window</p>
        <a href="{navigator_url}" target="_blank" class="btn-large">
            Open Multi-Layer Navigator
        </a>
    </div>
    
    <div class="instructions">
        <h3>💡 How to Use Navigator with Multiple Layers</h3>
        <ul>
            <li>Switch between layers using the <strong>tabs at the top</strong> of Navigator</li>
            <li>Use <strong>"Create Layer from other layers"</strong> to combine or compare layers</li>
            <li>Export modified layers or create custom views</li>
            <li>Use filtering and search to focus on specific techniques or tactics</li>
        </ul>
    </div>
    
    <div class="layer-list">
        <h3>📋 Included Layers</h3>
        {layer_items}
    </div>
    
    <div class="download-section">
        <h3>💾 Download Individual Layer Files</h3>
        <p style="margin-bottom: 15px; color: #666;">Download JSON files to manually upload or share:</p>
        {download_buttons}
    </div>
    
    <div class="footer">
        <p>Generated by ATT&CK Gap Analysis Framework</p>
    </div>
</body>
</html>
    """
    
    # Load all layers into a single array
    all_layers = []
    layer_items = []
    download_buttons = []
    
    for tab_name, layer_path in layers.items():
        # Read layer JSON
        with open(layer_path, 'r') as f:
            layer_json = json.load(f)
        
        all_layers.append(layer_json)
        
        # Get layer stats
        technique_count = len(layer_json.get('techniques', []))
        description = layer_json.get('description', 'No description')
        
        # Create layer list item
        layer_item = f"""
        <div class="layer-item">
            <h4>{tab_name}</h4>
            <p>{description[:150]}{'...' if len(description) > 150 else ''}</p>
            <div class="stats">📊 {technique_count} techniques</div>
        </div>
        """
        layer_items.append(layer_item)
        
        # Create download button
        layer_filename = layer_path.name
        download_button = f'<a href="{layer_filename}" download="{layer_filename}" class="btn">{tab_name.split()[0]} - {layer_filename}</a>'
        download_buttons.append(download_button)
    
    # Create Navigator URL with all layers as base64
    # Navigator supports loading an array of layers
    all_layers_json = json.dumps(all_layers)
    all_layers_b64 = base64.b64encode(all_layers_json.encode('utf-8')).decode('utf-8')
    navigator_url = f"https://mitre-attack.github.io/attack-navigator/#layerURL=data:application/json;base64,{all_layers_b64}"
    
    html = html_template.format(
        timestamp=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        layer_count=len(layers),
        navigator_url=navigator_url,
        layer_items='\n        '.join(layer_items),
        download_buttons='\n        '.join(download_buttons)
    )
    
    with open(output_file, 'w') as f:
        f.write(html)
    
    logger.info(f"✅ Multi-layer Navigator page created: {output_file}")

# Prepare layers for export
layers_to_export = {
    '🔥 Composite Threat': OUTPUT_DIR / 'composite_threat.json',
    '🛡️ Coverage': OUTPUT_DIR / 'coverage.json',
    '📊 Gap Analysis': OUTPUT_DIR / 'gap_analysis.json'
}

# Add individual threat actor layers
for i, actor in enumerate(threat_actors):
    layer_file = OUTPUT_DIR / f'actor_{i}_{actor.name.replace(" ", "_")}.json'
    actor.layer.save(layer_file)
    layers_to_export[f'👥 {actor.name}'] = layer_file

# Create multi-tab HTML
output_html = OUTPUT_DIR / 'gap_analysis_navigator.html'
create_multi_tab_navigator(layers_to_export, output_html)

print(f"✅ Navigator page created!")
print(f"📁 Output location: {output_html.absolute()}")
print(f"\n🌐 Open in browser: file://{output_html.absolute()}")
print(f"\n💡 Tip: Click 'Open Multi-Layer Navigator' to view all layers as tabs in one window!")

# ============================================================================
# MULTI-TAB NAVIGATOR EXPORT
# ============================================================================

"""
Generate Gap Analysis Report
Create markdown and/or HTML report with recommendations
"""

def generate_markdown_report(
    threat_actors: List[ThreatActor],
    composite_df: pd.DataFrame,
    priority_gaps: pd.DataFrame,
    stats: dict,
    output_file: Path
):
    """Generate comprehensive markdown report"""
    
    report = f"""# ATT&CK Gap Analysis Report

**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}  
**Analysis Version:** 1.0  
**ATT&CK Version:** {ATTACK_VERSION}

---

## Executive Summary

This gap analysis evaluates your organization's detection coverage against {len(threat_actors)} prioritized threat actors.

### Key Findings

- **Total Techniques Analyzed:** {stats.get('total_techniques', 0)}
- **Under-Covered Techniques:** {stats.get('under_covered_count', 0)} ({stats.get('under_covered_pct', 0):.1f}%)
- **Average Gap Score:** {stats.get('avg_gap', 0):.2f} / 10
- **Maximum Gap:** {stats.get('max_gap', 0):.2f}

### Gap Distribution

"""
    
    for category, count in stats.get('categories', {}).items():
        pct = (count / stats.get('total_techniques', 1)) * 100
        report += f"- **{category.upper()}:** {count} techniques ({pct:.1f}%)\n"
    
    report += f"""

---

## Threat Actor Profile

The following threat actors were included in this analysis:

"""
    
    for actor in threat_actors:
        report += f"""
### {actor.name} (Weight: {actor.weight})

{actor.description}

- **Techniques:** {len(actor.techniques)}
- **Average Technique Score:** {sum(t['score'] for t in actor.techniques.values()) / len(actor.techniques) if actor.techniques else 0:.1f} / 5

"""
    
    report += """

---

## Priority Gaps

The following techniques represent the highest priority gaps (threat presence with insufficient coverage):

"""
    
    if not priority_gaps.empty:
        report += priority_gaps.to_markdown(index=False)
    else:
        report += "*No significant gaps identified.*"
    
    report += """

---

## Top Composite Threats

The following techniques are most prevalent across your threat landscape:

"""
    
    top_threats = composite_df.head(10)
    report += top_threats[['Technique ID', 'Name', 'Composite Score', 'Actor Count']].to_markdown(index=False)
    
    report += """

---

## Recommendations

### Immediate Actions (Critical/High Gaps)

"""
    
    critical_gaps = priority_gaps[priority_gaps['Category'].isin(['CRITICAL', 'HIGH'])]
    
    if not critical_gaps.empty:
        for _, row in critical_gaps.head(5).iterrows():
            tech_id = row['Technique ID']
            metadata = attack_loader.get_technique_metadata(tech_id)
            
            report += f"""
#### {tech_id}: {row['Name']}

- **Gap Score:** {row['Gap Score']:.1f} / 10
- **Threat Score:** {row['Threat Score']}
- **Current Coverage:** {row['Coverage Score']} / 5
- **Tactics:** {row['Tactics']}

**Description:** {metadata['description'][:200]}...

**Recommended Actions:**
- Implement detection rules for this technique
- Ensure visibility into {', '.join(metadata['platforms'][:3])} platforms
- Review MITRE's detection guidance

---
"""
    else:
        report += "*No critical or high priority gaps identified.*\n"
    
    report += """

### Long-Term Strategy

1. **Enhance Detection Coverage:** Focus on techniques with gap scores > 5
2. **Data Source Assessment:** Ensure visibility into key platforms and data sources
3. **Regular Reassessment:** Update this analysis quarterly or after major infrastructure changes
4. **Threat Intelligence Integration:** Continuously update threat actor profiles

---

## Appendix

### Methodology

**Threat Scoring (1-5 scale):**
- 5: Signature/frequent technique for this actor
- 3: Moderately used technique
- 1: Occasionally observed

**Coverage Scoring (1-5 scale):**
- 5: Full detection with automated response
- 3: Manual detection capability
- 1: Minimal visibility

**Gap Scoring (-10 to +10 scale):**
- Positive scores indicate under-coverage (needs improvement)
- Negative scores indicate over-coverage relative to threat
- Gap = (Weighted Threat Score - Coverage Score) normalized to ±10 range

### Data Sources

- **MITRE ATT&CK:** {ATTACK_API['stix']}
- **Coverage Data:** DeTTECT framework
- **Threat Intelligence:** {len(threat_actors)} configured threat actors

---

*Report generated by ATT&CK Gap Analysis Framework*
"""
    
    with open(output_file, 'w') as f:
        f.write(report)
    
    logger.info(f"✅ Report saved: {output_file}")

# Generate report
report_file = OUTPUT_DIR / 'gap_analysis_report.md'
generate_markdown_report(
    threat_actors=threat_actors,
    composite_df=composite_df,
    priority_gaps=priority_gaps,
    stats=stats,
    output_file=report_file
)

print(f"✅ Gap analysis report generated!")
print(f"📄 Report location: {report_file.absolute()}")

# Optionally convert to HTML
try:
    import markdown
    with open(report_file, 'r') as f:
        md_content = f.read()
    html_content = markdown.markdown(md_content, extensions=['tables'])
    
    html_report = OUTPUT_DIR / 'gap_analysis_report.html'
    with open(html_report, 'w') as f:
        f.write(f"<html><head><style>body{{font-family:Arial;max-width:900px;margin:40px auto;}}table{{border-collapse:collapse;width:100%;}}th,td{{border:1px solid #ddd;padding:8px;text-align:left;}}th{{background:#f2f2f2;}}</style></head><body>{html_content}</body></html>")
    
    print(f"📄 HTML report: {html_report.absolute()}")
except ImportError:
    print("ℹ️  Install 'markdown' package for HTML export: pip install markdown")

# ============================================================================
# COMPLETE TTP DATA EXPORT
# ============================================================================

"""
Complete TTP Data Export to CSV
Export all techniques with threat scores, coverage scores, gaps, and annotations
"""

print("\n" + "=" * 60)
print("📊 Exporting complete TTP data to CSV...")
print("=" * 60)

# Build comprehensive TTP data
ttp_data = []

# Get all techniques from composite and gap analysis
all_tech_ids = set(composite_scores.keys()) | set(gap_scores.keys())

for tech_id in all_tech_ids:
    # Get metadata
    metadata = attack_loader.get_technique_metadata(tech_id)
    
    # Get scores
    threat_score = composite_scores.get(tech_id, {}).get('score', 0)
    coverage_score = coverage_scores.get(tech_id, 0)
    gap_data = gap_scores.get(tech_id, {})
    gap_score = gap_data.get('gap_score', 0)
    category = gap_data.get('category', 'N/A')
    
    # Get actors for this technique
    actors = composite_scores.get(tech_id, {}).get('actors', [])
    actors_str = ', '.join(actors) if actors else 'N/A'
    
    # Build row
    ttp_data.append({
        'Technique ID': tech_id,
        'Name': metadata['name'],
        'Tactics': ', '.join(metadata['tactics']) if metadata['tactics'] else 'N/A',
        'Platforms': ', '.join(metadata['platforms']) if metadata['platforms'] else 'N/A',
        'Threat Score': round(threat_score, 2),
        'Coverage Score': coverage_score,
        'Gap Score': round(gap_score, 2),
        'Category': category.upper(),
        'Threat Actors': actors_str,
        'Description': metadata['description'][:200] + '...' if len(metadata['description']) > 200 else metadata['description']
    })

# Create DataFrame
ttp_df = pd.DataFrame(ttp_data)

# Sort by gap score (highest to lowest)
ttp_df = ttp_df.sort_values('Gap Score', ascending=False)

# Save to CSV
csv_file = OUTPUT_DIR / 'complete_ttp_data.csv'
ttp_df.to_csv(csv_file, index=False)

print(f"✅ Complete TTP data exported!")
print(f"📁 CSV location: {csv_file.absolute()}")
print(f"📊 Total techniques: {len(ttp_df)}")
print(f"\nColumns included:")
print(f"   • Technique ID, Name, Tactics, Platforms")
print(f"   • Threat Score, Coverage Score, Gap Score")
print(f"   • Category, Threat Actors, Description")

# Display sample
print(f"\n📋 Sample data (top 5 gaps):")
display(ttp_df.head(5)[['Technique ID', 'Name', 'Threat Score', 'Coverage Score', 'Gap Score', 'Category']])


In [ ]:
"""
Generate Integrated Dashboard
Automatically create unified dashboard with Navigator and embedded report
"""

print("🎨 Generating integrated dashboard...")
print("=" * 60)

# Run the dashboard generator
try:
    exec(open('create_dashboard.py').read())
    print("\n" + "=" * 60)
    print("🎉 Dashboard generation complete!")
except FileNotFoundError:
    print("⚠️  create_dashboard.py not found, skipping dashboard generation")
    print("   The separate Navigator and report files are still available in output/")
except Exception as e:
    print(f"⚠️  Error generating dashboard: {e}")
    print("   The separate Navigator and report files are still available in output/")


## ✅ Analysis Complete!

### Generated Outputs

Your gap analysis is complete. The following files have been generated:

1. **Navigator Layers** (JSON):
   - `composite_threat.json` - Weighted threat landscape
   - `coverage.json` - Current detection coverage
   - `gap_analysis.json` - Coverage gaps
   - Individual threat actor layers

2. **Multi-Tab Viewer** (HTML):
   - `gap_analysis_navigator.html` - Interactive Navigator view

3. **Reports**:
   - `gap_analysis_report.md` - Detailed markdown report
   - `gap_analysis_report.html` - HTML version (if available)

### How to Use

1. **Open the Navigator:** Open `gap_analysis_navigator.html` in your browser to explore the multi-tab visualization
2. **Review Priority Gaps:** Check the report for prioritized recommendations
3. **Share with Stakeholders:** Export the HTML or PDF for presentations

### Next Steps

- **Update Coverage:** Add detections for high-priority gaps
- **Refine Weights:** Adjust threat actor weights based on current intelligence
- **Periodic Updates:** Re-run this analysis quarterly
- **Expand Coverage Data:** Enhance your DeTTECT YAML with more coverage details

### Customization

To modify this analysis:
- Edit `THREAT_ACTORS` configuration in Cell 8
- Update `DETTECT_YAML_PATH` with your coverage data
- Adjust weights and scoring in the configuration cells
- Re-run from Cell 9 onwards

---

**Questions or issues?** Check the technical specification or reach out for support.